# 📘 Notebook 16: Exploring the Data with Spark

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module E: A Real Warehouse, End to End · Notebook 4 of 6 in this module · (16 of 18 overall)*

---

> 🧭 **How to use this notebook.** Run the cells in order, from top to bottom: click a cell and press **Shift+Enter**. After most cells there is a short paragraph headed **What you should see**: compare it with your own output before you move on. If your numbers differ, stop and find out why, because that is how data work is learned. The exercises at the end start easy and get harder: first **warm-up** questions, then **think it through** questions (what would you run, and why?), then **technical** tasks. Try each one before you open its solution.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Start Spark from Python, and explain **lazy** evaluation: transformations that wait, actions that run
- Read CSV files into Spark, and say what `inferSchema` costs and when to write the schema yourself
- Join a star schema into one **wide table** in Spark, and read the **join strategy** in the query plan
- Ask the same question with the **DataFrame API** and with **Spark SQL**
- Carry out an **exploratory data analysis**: distributions, missing values, outliers and relationships
- Keep the heavy work in Spark and bring only **small results** back to pandas for plotting

> **Prerequisites:** Notebook 14 (this warehouse). Notebooks 09 and 10 introduced Spark, but this notebook starts from the beginning. Plotting with Matplotlib is in *From Python to Deep Learning and Fine-Tuning*, Notebook 07.
>
> 🔭 **Why this notebook matters:** *exploratory data analysis* (EDA) is the conversation an analyst has with a data set before trusting it with a decision: how is each variable distributed, what is missing, what is strange, what moves together. On a laptop it is done in pandas. When the data lives in a cluster, the same conversation has to be held through **Spark**, and one rule becomes essential: let Spark compute over all the rows, and bring back to your own machine only what fits on a chart. This notebook holds that conversation with the AdventureWorks sales.

> 📦 **Libraries used in this notebook.**
>
> - **PySpark**, in local mode: the cores of this machine play the part of a cluster.
> - **pandas** and **Matplotlib**, for the small results and the charts.
>
> The cell below installs PySpark. In Google Colab, PySpark and the Java it needs are already present and the cell finishes in a moment.

> 🗃️ **The data.** This module works with **AdventureWorks**, the sample data warehouse that Microsoft publishes for learning SQL Server: the online sales of an imaginary bicycle company, with its customers, products and dates. It is open data (MIT licence) and much closer to a real warehouse than anything we could invent. The cell below downloads it from Microsoft's GitHub repository (about 17 MB, once per session) and saves each table as an ordinary CSV file.

In [ ]:
%pip install -q pyspark

In [ ]:
import io
import os
import re
import urllib.request
import zipfile

import pandas as pd

AW_URL = ("https://github.com/microsoft/sql-server-samples/releases/download/"
          "adventureworks/AdventureWorksDW-data-warehouse-install-script.zip")
AW_TABLES = ["FactInternetSales", "DimCustomer", "DimGeography", "DimSalesTerritory", "DimProduct",
             "DimProductSubcategory", "DimProductCategory", "DimDate", "DimPromotion", "DimCurrency"]


def fetch_adventureworks(folder="adventureworks_dw"):
    """Download Microsoft's AdventureWorks sample warehouse once, and save each table as a CSV file with a header."""
    if os.path.isdir(folder):
        return folder
    archive = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(AW_URL).read()))
    script = archive.read("instawdbdw.sql")
    script = script.decode("utf-16" if script[:2] in (b"\xff\xfe", b"\xfe\xff") else "utf-8-sig")
    os.makedirs(folder)
    for table in AW_TABLES:
        # The column names are not in the data files: they are in the CREATE TABLE statements of the install script.
        body = re.search(r"CREATE TABLE \[dbo\]\.\[" + table + r"\]\((.*?)\n\)", script, re.S).group(1)
        columns = re.findall(r"^\s*\[(\w+)\]", body, re.M)
        text = archive.read(table + ".csv").decode("utf-8-sig")
        df = pd.read_csv(io.StringIO(text), sep="|", header=None, names=columns, dtype=str, keep_default_na=False)
        df = df.drop(columns=[c for c in ("LargePhoto", "SalesTerritoryImage") if c in df])   # pictures as hex text
        df.replace("", None).to_csv(f"{folder}/{table}.csv", index=False)
    return folder


def load(name, folder=None):
    """Read one table into pandas. Other exports of this warehouse are sometimes saved as Latin-1, so try both."""
    path = f"{folder or DATA}/{name}.csv"
    try:
        return pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
    except UnicodeDecodeError:
        return pd.read_csv(path, encoding="latin-1", low_memory=False)


DATA = fetch_adventureworks()
print("Tables:", sorted(f[:-4] for f in os.listdir(DATA)))

**What you should see:** the list of ten table names, as in the earlier notebooks.

In [ ]:
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types as T

os.environ.setdefault("PYSPARK_PYTHON", sys.executable)   # the same Python for the driver and the workers

## 1. Spark in plain words

### Intuition first
pandas keeps a whole table in the memory of **one** machine and works on it at once. **Spark** is built for tables too large for that. It cuts a table into pieces called **partitions** and gives them to many workers, which may sit on many machines (a **cluster**). Your notebook is the **driver**: it describes the work, Spark splits it into tasks, and the workers carry them out in parallel.

Here there is no cluster, so we use `local[*]`: *run on this machine, with one worker per processor core*. The code is exactly the code you would run on a cluster; only that one line would change.

A **SparkSession** is the entrance to Spark, like the connection in SQLite.

In [ ]:
spark = (SparkSession.builder
         .master("local[*]")                                   # this machine, all its cores
         .appName("adventureworks")
         .config("spark.sql.shuffle.partitions", "8")          # small data: 8 pieces after a shuffle are enough
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")                         # show only real errors
print("Spark", spark.version, "with", spark.sparkContext.defaultParallelism, "cores")

**What you should see:** `Spark 3.x.x with 2 cores` in a free Colab session (a laptop usually shows more). The first start takes some seconds, because Spark launches a Java process in the background.

### Lazy evaluation: Spark waits until you ask for an answer
Spark commands come in two kinds:

* **transformations** describe a new table: `select`, `filter`, `withColumn`, `join`, `groupBy`. They return immediately and compute **nothing**;
* **actions** ask for a result: `count`, `show`, `collect`, `toPandas`. Only then does Spark plan the whole chain and run it.

This is called **lazy evaluation**. It lets Spark see the whole job before starting and choose the cheapest way to do it. Watch it happen on ten million numbers:

In [ ]:
numbers = spark.range(10_000_000)                       # a table with one column, id = 0, 1, 2, ...

started = time.perf_counter()
evens = numbers.filter(F.col("id") % 2 == 0)           # transformation: nothing runs yet
squares = evens.withColumn("square", F.col("id") * F.col("id"))
print(f"two transformations: {time.perf_counter() - started:.3f} s")

started = time.perf_counter()
print("rows:", squares.count())                         # action: now everything runs
print(f"one action:          {time.perf_counter() - started:.3f} s")
squares.show(3)

**What you should see:** the two transformations take a fraction of a second, because nothing was computed (most of that is Python talking to Java for the first time). The action takes several times longer, a few seconds, and prints `rows: 5000000`. Then `show(3)` prints the first three rows: `id` 0, 2, 4 and `square` 0, 4, 16.

Three things to remember from this cell:

* `F.col("id")` means *the column called id*. The module `F` (`pyspark.sql.functions`) holds Spark's functions on columns.
* `show(n)` prints a few rows **on the driver**; `count()` returns a number; `toPandas()` brings the **whole** table back as a pandas DataFrame, so use it only on small results.
* If a cell with only transformations runs instantly, that is not a sign that it worked. Errors in a transformation often appear only at the next action.

## 2. Reading the tables, and the cost of guessing types

### Intuition first
A CSV file is only text. Spark can **guess** each column's type (`inferSchema=True`), but to guess it must read the whole file once before reading it again for real. On a few megabytes that is nothing. On a terabyte it doubles the cost of every load, and a wrong guess (a postal code read as a number, losing its leading zero) can corrupt the data silently. So production pipelines **declare** the schema: a list of columns and their types (`IntegerType`, `DoubleType` for decimals, `StringType` for text). Let us measure the difference on the largest table.

In [ ]:
path = f"{DATA}/FactInternetSales.csv"

started = time.perf_counter()
guessed = spark.read.csv(path, header=True, inferSchema=True)
guessed.count()
t_guess = time.perf_counter() - started

# Declare the schema: take the column names, and say which are integers, decimals and text.
names = guessed.columns
integers = {"ProductKey", "OrderDateKey", "DueDateKey", "ShipDateKey", "CustomerKey", "PromotionKey", "CurrencyKey",
            "SalesTerritoryKey", "SalesOrderLineNumber", "RevisionNumber", "OrderQuantity"}
texts = {"SalesOrderNumber", "CarrierTrackingNumber", "CustomerPONumber", "OrderDate", "DueDate", "ShipDate"}
schema = T.StructType([T.StructField(n, T.IntegerType() if n in integers else
                                        T.StringType() if n in texts else T.DoubleType()) for n in names])
started = time.perf_counter()
declared = spark.read.csv(path, header=True, schema=schema)
declared.count()
t_declared = time.perf_counter() - started
print(f"inferSchema: {t_guess:.2f} s   declared schema: {t_declared:.2f} s")
declared.printSchema()

**What you should see:** the declared read is faster than the guessed one, often by several times (about 5.6 s against 0.7 s on a laptop) (part of the first time is also Spark warming up), and then the schema: 26 columns, each with the type we declared, such as `ProductKey: integer` and `SalesAmount: double`.

For the small dimension tables we let Spark guess; for the fact table we keep the declared schema. We also turn the text `OrderDate` into a real date and add the year and month as columns, with `withColumn(name, expression)`.

In [ ]:
def read(name):
    return spark.read.csv(f"{DATA}/{name}.csv", header=True, inferSchema=True)

sales = (declared
         .withColumn("OrderDate", F.to_date(F.substring("OrderDate", 1, 10)))
         .withColumn("Year", F.year("OrderDate"))
         .withColumn("Month", F.month("OrderDate")))
product, subcategory, category = read("DimProduct"), read("DimProductSubcategory"), read("DimProductCategory")
customer, geography = read("DimCustomer"), read("DimGeography")
sales.select("SalesOrderNumber", "OrderDate", "Year", "Month", "SalesAmount").show(3)

**What you should see:** three sales from 2010-12-29, with `Year` 2010, `Month` 12 and amounts of 3578.27 and 3399.99.

## 3. One wide table

### Intuition first
For exploration, analysts prefer one **wide table** to a star: every row a sale, every column something about it. Building it takes five joins. On a cluster a join is the expensive operation, because matching rows may live on different machines and must be **shuffled** across the network. Unless one side is small: then Spark copies the small table to every worker, a **broadcast join**, and nothing else moves. All our dimensions are small.

`join(other, "Key", "left")` keeps every row of the left table, even when the partner is missing. That is the safe choice for building a wide table: no sale disappears.

In [ ]:
prod = (product.select("ProductKey", "EnglishProductName", "ProductSubcategoryKey")
        .join(subcategory.select("ProductSubcategoryKey", "EnglishProductSubcategoryName", "ProductCategoryKey"),
              "ProductSubcategoryKey", "left")
        .join(category.select("ProductCategoryKey", "EnglishProductCategoryName"), "ProductCategoryKey", "left"))
cust = (customer.select("CustomerKey", "GeographyKey", "BirthDate", "Gender", "MaritalStatus", "YearlyIncome",
                        "TotalChildren", "EnglishEducation", "EnglishOccupation", "HouseOwnerFlag",
                        "NumberCarsOwned", "CommuteDistance")
        .join(geography.select("GeographyKey", "EnglishCountryRegionName"), "GeographyKey", "left"))

wide = (sales.join(prod, "ProductKey", "left").join(cust, "CustomerKey", "left")
        .withColumnRenamed("EnglishProductCategoryName", "Category")
        .withColumnRenamed("EnglishProductSubcategoryName", "Subcategory")
        .withColumnRenamed("EnglishCountryRegionName", "Country")
        .withColumn("Margin", F.col("SalesAmount") - F.col("TotalProductCost")))
wide.cache()                                    # keep it in memory: every question below reads it
print(wide.count(), "rows,", len(wide.columns), "columns")

**What you should see:** `60398 rows, 46 columns`: the same number of rows as the fact table, so the left joins neither lost nor duplicated a sale, and many more columns.

`cache()` asks Spark to keep the table in memory after the next action. Without it, every question below would redo the five joins from the CSV files, because Spark is lazy and remembers only *how* to build `wide`, not the result.

Which kind of join did Spark choose? The **query plan** says. The cell below counts the join types in it, before and while running.

In [ ]:
plan = wide._jdf.queryExecution().executedPlan().toString()
final, _, initial = plan.partition("== Initial Plan ==")
for label, part in [("plan before running", initial), ("plan actually run", final)]:
    print(f"{label:<20} broadcast joins: {part.count('BroadcastHashJoin')}   sort-merge joins: {part.count('SortMergeJoin')}")

**What you should see:** before running, 3 broadcast joins and 2 sort-merge joins; in the plan actually run, 5 broadcast joins and 0 sort-merge joins.

Before running, Spark could only estimate the size of the joined tables, so it planned some joins as **sort-merge** joins, the expensive kind that shuffles both sides. While running, it measured them, found them small, and switched to **broadcast** joins. This re-planning during execution is called **adaptive query execution**, and it is switched on by default since Spark 3.2.

## 4. One question, two languages

Spark understands two languages for the same work. The **DataFrame API** chains methods in Python: `groupBy` to make groups, `agg` to summarise each group (`F.count`, `F.sum`, ...), `alias` to name the result, `orderBy` to sort. **Spark SQL** accepts ordinary SQL, as in Notebook 14, once the table has a name. Both are turned into the same plan, so choose whichever reads better.

First, revenue by year and category with the DataFrame API:

In [ ]:
(wide.groupBy("Year", "Category")
     .agg(F.count("*").alias("lines"), F.round(F.sum("SalesAmount"), 0).alias("revenue"))
     .orderBy("Year", F.desc("revenue"))
     .show())

**What you should see:** ten rows. Until 2011 only bikes; in 2012 the first accessories (106 lines) and clothing (22 lines) appear; in 2013 accessories have 34,409 lines against 9,706 for bikes, yet bikes bring about 15.4 million (shown as `1.5359502E7`, scientific notation for 15,359,502) against about 0.67 million for accessories. 2014 has no bikes at all.

Now SQL. `createOrReplaceTempView` gives the DataFrame a name that SQL can use:

In [ ]:
wide.createOrReplaceTempView("sales_wide")
spark.sql('''
    SELECT Country, ROUND(SUM(SalesAmount)) AS revenue, COUNT(DISTINCT CustomerKey) AS customers,
           ROUND(100 * SUM(Margin) / SUM(SalesAmount), 1) AS margin_pct
    FROM sales_wide
    GROUP BY Country
    ORDER BY revenue DESC
''').show()

**What you should see:** six countries. The United States (about 9.4 million from 7,819 customers) and Australia (about 9.1 million from only 3,591 customers) lead; Canada is last with about 2.0 million. The margin is between 40.7% and 42.0% everywhere: the countries differ in how much they buy, not in how profitable each sale is.

Notice that Australia earns almost as much as the United States with fewer than half the customers. That is a question worth asking next; Exercise 8 gives one part of the answer.

## 5. Exploratory data analysis

EDA asks four questions of every variable that matters: how is it **distributed**, what is **missing**, what is **unusual**, and what is it **related** to. Spark answers each over all rows; we bring back only the answer.

### 5.1 Summary statistics and missing values
`summary()` is Spark's `describe()`: count, mean, standard deviation, minimum, quartiles and maximum of each column, computed over all rows.

In [ ]:
wide.select("SalesAmount", "Margin", "Freight", "YearlyIncome", "TotalChildren").summary(
    "count", "mean", "stddev", "min", "25%", "50%", "75%", "max").show()

**What you should see:** a table with 8 statistics for 5 columns. `SalesAmount` has a mean of about 486 but a median (`50%`) of only 29.99 and a maximum of 3,578.27: many cheap lines and a few expensive ones, as in Notebook 13. `YearlyIncome` runs from 10,000 to 170,000 with a median of 60,000, and `TotalChildren` from 0 to 5.

Next, the number of missing values (`null` in Spark) in every column, computed in **one** pass: for each column, turn *is null* into 1 or 0 and add up. The answer is one row, which we bring back with `toPandas()`.

In [ ]:
missing = wide.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in wide.columns]).toPandas().T[0]
missing[missing > 0]

**What you should see:** only `CarrierTrackingNumber` and `CustomerPONumber`, each with 60,398 nulls: the two columns that Notebook 13 found entirely empty. The whole-table check came back as a single row. That is the pattern of this notebook: compute over everything, bring back little.

### 5.2 The unit of analysis

The order line is the right unit for questions about products. For questions about **people** the unit is the **customer**, so we aggregate to one row per customer: total spend, number of orders, age, and days since the last order. That table has 18,484 rows, small enough to bring back to pandas (`cpd`, *customers in pandas*).

In [ ]:
last_day = sales.agg(F.max("OrderDate")).first()[0]
customers = (wide.groupBy("CustomerKey", "Gender", "MaritalStatus", "YearlyIncome", "EnglishEducation", "Country",
                          "BirthDate", "TotalChildren", "NumberCarsOwned", "HouseOwnerFlag", "CommuteDistance")
             .agg(F.round(F.sum("SalesAmount"), 2).alias("spend"),
                  F.countDistinct("SalesOrderNumber").alias("orders"),
                  F.max("OrderDate").alias("last_order"))
             .withColumn("age", F.floor(F.months_between(F.lit(last_day), F.to_date("BirthDate")) / 12))
             .withColumn("days_since_last", F.datediff(F.lit(last_day), "last_order")))
cpd = customers.toPandas()
print(cpd.shape)
cpd[["spend", "orders", "age", "days_since_last"]].describe().round(1)

**What you should see:** `(18484, 16)`, one row per customer, and a summary. The spend runs from 2.3 to about 13,300, with a mean of about 1,588 but a median of only 270. Half of the customers placed exactly one order (the median of `orders` is 1, the maximum 28). Ages run from 27 to 97, half of them between 35 and 52.

### 5.3 Distributions
A **histogram** cuts a variable into bins and counts the values in each. We draw three: the spend per customer, the same on a logarithmic scale (`log10`: 10 becomes 1, 100 becomes 2, 1,000 becomes 3), and the age.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
cpd["spend"].plot.hist(bins=60, ax=ax[0], title="Spend per customer")
np.log10(cpd["spend"]).plot.hist(bins=60, ax=ax[1], title="Spend per customer, log10 scale")
cpd["age"].plot.hist(bins=30, ax=ax[2], title="Age at the last order date")
plt.tight_layout()
plt.show()

**What you should see:** on the left, a tall bar of over 9,000 customers in the first bin and a long thin tail to the right: the spend is **heavily skewed**. In the middle, on the log scale, the picture separates into **two humps**: customers who bought only accessories or clothing (spend in the tens, around 1 to 2.5 on the log axis) and customers who bought a bicycle (spend in the thousands, around 3 to 4). On the right, ages from the late twenties to the nineties, most between about 30 and 55.

For a skewed variable the **median** (the middle value) is a more honest summary than the mean:

In [ ]:
print(f"mean spend {cpd['spend'].mean():.0f}   median spend {cpd['spend'].median():.0f}")

**What you should see:** `mean spend 1588   median spend 270`. The mean is almost six times the median: a few big spenders pull it up. A report that said *"the typical customer spends about 1,600"* would be wrong for most customers.

### 5.4 What is unusual?

The **interquartile range** rule marks values far beyond the middle half of the data. Q1 and Q3 are the values below which a quarter and three quarters of the customers lie; anything above Q3 + 1.5 × (Q3 minus Q1) is flagged. Spark computes quantiles on the full data with `approxQuantile`, which accepts a tiny, controllable error (here 0.001) in return for not sorting everything.

In [ ]:
q1, q3 = customers.approxQuantile("spend", [0.25, 0.75], 0.001)
upper = q3 + 1.5 * (q3 - q1)
flagged = customers.filter(F.col("spend") > upper).count()
print(f"Q1 = {q1:.0f}   Q3 = {q3:.0f}   upper fence = {upper:.0f}   customers above it: {flagged}")

**What you should see:** `Q1 = 50   Q3 = 2498   upper fence = 6170   customers above it: 677`. A quarter of the customers spent 50 or less, a quarter spent about 2,500 or more, and 677 customers spent more than 6,170.

An outlier is a question, not an error. These are the company's best customers. Removing them would make the averages look tidier and the analysis wrong. The decision (keep, cap, or study separately) depends on the purpose, and it should always be written down.

### 5.5 Relationships
A **box plot** shows, for each group, the middle half of the values as a box and the median as a line. A **correlation** (from -1 to 1) measures how strongly two numerical variables rise or fall together; 0 means no straight-line relationship.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
cpd.boxplot(column="spend", by="EnglishEducation", ax=ax[0], showfliers=False)
ax[0].set_title("Spend by education (outliers hidden)")
ax[0].tick_params(axis="x", rotation=25)
cpd.groupby("YearlyIncome")["spend"].median().plot(marker="o", ax=ax[1], title="Median spend by yearly income")
plt.suptitle("")
plt.tight_layout()
plt.show()

**What you should see:** on the left, one box per education level. Customers with a bachelor's or graduate degree have the tallest boxes (their top quarter spends the most), while customers with partial high school have the shortest and lowest box. The medians of the first four groups are close to each other. On the right, the median spend jumps up and down from one income level to the next, but the higher incomes (150,000 and above) clearly spend more. Real relationships are rarely smooth lines.

In [ ]:
cpd[["spend", "orders", "age", "YearlyIncome", "TotalChildren", "NumberCarsOwned", "days_since_last"]].corr().round(2)

**What you should see:** a 7 by 7 table (scroll to see all columns). The strongest link of spend is with the number of orders (0.38), close to arithmetic: more orders, more spend. Spend and income: 0.16, weak. Spend and age: -0.06, almost nothing. The largest value in the table is between income and the number of cars (0.47), which says something about the customers, not about their spending.

Weak correlations are a finding too: they warn that no single attribute will predict spending well, which matters for the next notebook.

### 5.6 Change over time

Bring back one row per month and category, a few dozen rows, and plot them. `date_trunc("month", ...)` turns every date into the first day of its month.

In [ ]:
monthly = (wide.groupBy(F.date_trunc("month", "OrderDate").alias("month"), "Category")
               .agg(F.sum("SalesAmount").alias("revenue"))
               .toPandas()
               .pivot(index="month", columns="Category", values="revenue"))
monthly.plot(figsize=(12, 4), title="Revenue per month and category", logy=True)
plt.show()

**What you should see:** three lines on a logarithmic axis (each step up is ten times more). Bikes run from the end of 2010 to December 2013; accessories and clothing start at the very end of 2012 and quickly settle at their own, lower levels; and at the far right the accessories and clothing lines drop sharply in the last month.

The drop is not a crisis: January 2014 is the last, incomplete month of the data. An analysis that forgot it would report a collapse.

## ⚠️ Common pitfalls

* **Calling `toPandas()` or `collect()` on a big table.** Everything travels to the driver, which may run out of memory. Aggregate first.
* **Thinking a cell worked because it ran instantly.** Transformations are lazy; errors appear at the next action.
* **Forgetting `cache()` on a table you read many times**, so Spark recomputes it every time; or caching everything, and filling the memory.
* **Using an inner join to build a wide table** when some keys may be missing: rows disappear silently. Compare the row counts.
* **Trusting `inferSchema`** for codes with leading zeros, such as postal codes and phone numbers.

## 6. Where we stand

| Step of the EDA | How it was done in Spark | What came back |
|---|---|---|
| Load with the right types | a declared schema for the big table | nothing |
| Combine the star | broadcast joins into one cached wide table | nothing |
| Summaries, missing values | `summary()`, a null count per column | one small table each |
| Distributions, relationships | aggregate to customers, then pandas | 18,484 rows |
| Outliers | `approxQuantile` on all rows | three numbers |
| Trends | aggregate by month and category | a few dozen rows |

The pattern is always the same: compute over everything where the data lives, and move only answers. The next notebook keeps the same customers and asks Spark to learn from them.

---
## ✏️ Exercises

## Level 1: warm-up

One Spark command each: `filter`, `groupBy`, `count`, `show`, or a short SQL query.

### Exercise 1 (Lines per country)
With the DataFrame API, count the order lines per `Country` in `wide`, largest first.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
wide.groupBy("Country").count().orderBy(F.desc("count")).show()
```

*The United States has the most lines (21,344), then Australia (13,345); France the fewest (5,558). groupBy followed by count is the most common question in Spark.*

</details>

### Exercise 2 (Customers per year, in SQL)
With Spark SQL on `sales_wide`, count the **distinct** customers per `Year`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
spark.sql("SELECT Year, COUNT(DISTINCT CustomerKey) AS customers FROM sales_wide GROUP BY Year ORDER BY Year").show()
```

*14 customers in 2010, 2,216 in 2011, 3,255 in 2012, 17,429 in 2013 and 834 in the four weeks of 2014. COUNT(DISTINCT ...) counts people, not lines.*

</details>

### Exercise 3 (The most expensive lines)
Show the five order lines of 2013 with the highest `SalesAmount`: their order number, product and amount. (Use `filter`, `orderBy(F.desc(...))` and `show(5)`.)

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
(wide.filter(F.col("Year") == 2013)
     .select("SalesOrderNumber", "EnglishProductName", "SalesAmount")
     .orderBy(F.desc("SalesAmount"))
     .show(5, truncate=False))
```

*Five Road-250 Black bikes at 2,443.35 each. The most expensive bike of all (3,578.27) does not appear, because it was no longer sold in 2013: the filter, not the data, decides what you see.*

</details>

## Level 2: think it through

These questions are about **reasoning**. Decide first what you would need to see, then write the code that shows it.

### Exercise 4 (Lazy or slow?)
A colleague says: *"My Spark cell with five joins ran in 0.01 seconds, so joins are cheap."* **What would you run to show them what really happened?** Time the definition of a new filtered table and then its `count()`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
started = time.perf_counter()
big_2013 = wide.filter((F.col("Year") == 2013) & (F.col("SalesAmount") > 1000))
print(f"definition: {time.perf_counter() - started:.4f} s")
started = time.perf_counter()
print("rows:", big_2013.count())
print(f"count():    {time.perf_counter() - started:.4f} s")
```

*The definition took a few hundredths of a second, the count about half a second and found 6,718 lines: nothing ran until the action. The colleague timed a description of the work, not the work. To time a Spark job, time an action.*

</details>

### Exercise 5 (Can a join create rows?)
The wide table has exactly as many rows as the fact table. **When could a left join produce *more* rows than the left table?** Check the condition that guarantees it cannot happen here, for the product and the customer tables.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for name, table, key in [("products", prod, "ProductKey"), ("customers", cust, "CustomerKey")]:
    print(f"{name:<10} rows {table.count():>6}   distinct {key} {table.select(key).distinct().count():>6}")
print("fact rows:", sales.count(), "  wide rows:", wide.count())
```

*A left join adds rows when a key appears more than once in the right table: each sale is then copied once per match. Here every product key and every customer key appears exactly once (606 and 18,484), so the wide table keeps exactly 60,398 rows. Checking key uniqueness before a join prevents silently doubled revenue.*

</details>

### Exercise 6 (How approximate is approximate?)
`approxQuantile` accepts an error. **Before you trust a fast approximate median, what would you compare it with?** Compute the median spend with relative error 0.05, with 0.001, and exactly in pandas (`cpd['spend'].median()`), and time the two Spark calls.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
for err in (0.05, 0.001):
    started = time.perf_counter()
    m = customers.approxQuantile("spend", [0.5], err)[0]
    print(f"error {err}: median {m:.2f}  in {time.perf_counter() - started:.2f} s")
print(f"exact (pandas): {cpd['spend'].median():.2f}")
```

*With error 0.05 the median comes out at about 228, some 15% below the exact 270; with 0.001 it is about 265, close. The error is in rank: 0.05 allows any value between the 45th and the 55th percentile, and spend changes fast around its middle. Always check an approximation once against an exact answer on data you can afford, then choose the error with the distribution in mind.*

</details>

## Level 3: technical

Longer tasks that combine several steps.

### Exercise 7 (Margins by subcategory)
With Spark SQL on `sales_wide`, list the ten subcategories with the highest **margin percentage**, together with their revenue. Do the most profitable subcategories bring much revenue?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
spark.sql('''
    SELECT Subcategory, ROUND(SUM(SalesAmount)) AS revenue,
           ROUND(100 * SUM(Margin) / SUM(SalesAmount), 1) AS margin_pct
    FROM sales_wide GROUP BY Subcategory ORDER BY margin_pct DESC LIMIT 10
''').show()
```

*Ten subcategories share exactly the same margin, 62.6 percent: in this sample data the cost of many accessories was evidently set as a fixed share of the price. A value repeated that exactly is worth noticing in any data set, because real data is rarely so tidy. Their revenue is small next to the bicycles: a margin ranking and a revenue ranking answer different business questions.*

</details>

### Exercise 8 (Repeat customers by country)
Using `customers` (the Spark DataFrame), compute for each country the share of customers who placed **more than one** order. Return a small pandas table sorted by that share.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
(customers.groupBy("Country")
          .agg(F.round(100 * F.avg((F.col("orders") > 1).cast("int")), 1).alias("repeat_pct"),
               F.count("*").alias("customers"))
          .orderBy(F.desc("repeat_pct"))
          .toPandas())
```

*Australia stands out: 71.9% of its customers ordered more than once, against 22.2% in the United States. That explains part of the question from Section 4: fewer Australian customers, but they come back. The average of a 0/1 column is a share; computing it in Spark and returning six rows is the pattern of the whole notebook.*

</details>

### Exercise 9 (Plot without collecting, a little harder)
A histogram of `SalesAmount` over all **order lines** would need every value. Instead, let Spark count the lines in 20 equal-width bins of `log10(SalesAmount)` and bring back only the 20 counts. Plot them as a bar chart.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
lo, hi = wide.select(F.min(F.log10("SalesAmount")), F.max(F.log10("SalesAmount"))).first()
width = (hi - lo) / 20
bins = (wide.select(F.least(F.floor((F.log10("SalesAmount") - lo) / width), F.lit(19)).alias("bin"))
            .groupBy("bin").count().orderBy("bin").toPandas())
bins["from"] = (lo + bins["bin"] * width).round(2)
bins.plot.bar(x="from", y="count", figsize=(10, 3), legend=False, title="Order lines by log10(SalesAmount)")
plt.show()
```

*Twenty numbers travel instead of sixty thousand, or sixty billion. This is how dashboards on large data draw their histograms.*

</details>

## 🔑 Key takeaways

- Spark splits a table into **partitions** processed in parallel; the notebook is the **driver**.
- **Transformations** are lazy; **actions** (`count`, `show`, `collect`, `toPandas`) run the work.
- Declare the **schema** of large files: `inferSchema` reads the data twice and can guess wrong.
- Joining a big table to small ones uses **broadcast joins**, which move almost nothing; `cache()` a table you will read many times.
- The DataFrame API and Spark SQL are two languages for the same plans.
- EDA asks about **distributions**, **missing values**, **outliers** and **relationships**; choose the right **unit of analysis** first.
- Compute where the data lives and bring back only **answers**: aggregates, quantiles, binned counts.

**Next:** *Notebook 17: Machine Learning with Spark MLlib*, where the customer table becomes the input of models trained by Spark.

In [ ]:
spark.stop()

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*